# 05 · Differential expression: each disease against healthy, and diseases against each other

**Run order:** 02 facility CSVs → 03 select samples → 04 batch PCA (optional, to look at batch) → **05 (this notebook)**. Kernel: **R (WGCNA)**.
This is the equivalent of the repo's notebook 41, rebuilt to read notebook 03's CSVs and to include the **healthy controls**.

**Counts.** The **raw** counts of all eligible samples (`rnaseq_counts_raw_eligible.csv`), *not* the ComBat-seq counts of notebook 04. Batch (the facility) enters the model as a term instead, as the authors of limma and DESeq2 advise (Love MI et al. *F1000Research* 2015;4:1070).

**Method.** limma-voom (Law CW et al. *Genome Biol* 2014;15:R29) with the **patient as a block**: `duplicateCorrelation` estimates how alike two samples of the same patient are, and the model uses that (Smyth GK et al. *Bioinformatics* 2005;21:2067-2075). A patient seen at two facilities has one `patient_key`, so the block works across facilities too.

**Design.** `~ 0 + diagnosis + batch`: one coefficient per diagnosis (healthy controls are one of them, `HC`) and one per batch except the first.

**Comparisons.** Every disease against `HC` (positive log fold change = higher in the disease), and every pair of diseases.

**Top genes.** Benjamini-Hochberg adjusted p < 0.05 and a fold change of at least 2 (|log2 fold change| ≥ 1).

**How to read the healthy-control comparisons.** The healthy controls were sequenced at the same facilities as the patients, so the batch term compares like with like *where a facility holds both groups*. The summary table has a column `batches_with_both_groups`: the number of batches that contain samples of both groups in the comparison. If it is **0**, that comparison cannot be separated from batch at all and the notebook marks it. The more batches share both groups, the more the result is a within-batch comparison.

**Reads:** `rnaseq_out_v2/rnaseq_counts_raw_eligible.csv`, `rnaseq_samples_eligible.csv` (from 03).
**Writes (all CSV):** `rnaseq_out_v2/differential_expression/`: `summary_comparisons.csv`, and for each comparison `de_<A>_vs_<B>.csv` (all genes) and `top_genes_<A>_vs_<B>.csv`.

In [ ]:
RESULTS <- "/home/jovyan/session_data/Attempt0/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
OUT_DIR <- file.path(IN_DIR, "differential_expression")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

USE_ONE_PER_PATIENT <- FALSE   # FALSE = all eligible samples with the patient as a block (recommended); TRUE = one sample per patient, no block
MIN_PER_GROUP <- 3             # diagnoses with fewer samples than this are left out
ALPHA <- 0.05                  # adjusted p-value cut-off for the top genes
LFC   <- 1                     # |log2 fold change| cut-off for the top genes (1 = a fold change of 2)
RECODE <- c("Undifferentiated Interferonopathy" = "UAID")   # spellings merged into one diagnosis
GROUP_ORDER <- c("HC", "AGS", "CANDLE", "SAVI", "UAID")      # HC first, so every contrast reads disease minus healthy

if (!requireNamespace("edgeR", quietly = TRUE)) {            # one-time install if missing (notebook 04 does the same)
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
  BiocManager::install("edgeR", ask = FALSE, update = FALSE)
}
suppressMessages({library(limma); library(edgeR)})

## Read the eligible samples
Counts (genes × samples) and the sample table from notebook 03, put in the same order. Ensembl versions are removed. Diagnoses are tidied (`RECODE`), and any diagnosis with fewer than `MIN_PER_GROUP` samples is left out, because one or two samples cannot be compared with anything.

In [ ]:
cd <- read.csv(file.path(IN_DIR, "rnaseq_counts_raw_eligible.csv"), check.names = FALSE, row.names = 1)
gene_name <- cd$Gene_Name
counts <- as.matrix(cd[, colnames(cd) != "Gene_Name"]); storage.mode(counts) <- "numeric"
s <- read.csv(file.path(IN_DIR, "rnaseq_samples_eligible.csv"), check.names = FALSE, colClasses = "character")
stopifnot(!anyNA(match(colnames(counts), s$sample_id)))
s <- s[match(colnames(counts), s$sample_id), ]

ens <- sub("\\.[0-9]+$", "", rownames(counts))
stopifnot(!anyDuplicated(ens), !anyNA(counts), all(counts >= 0), all(counts == round(counts)))
rownames(counts) <- ens
keep_g <- !startsWith(ens, "ENSGR") | rowSums(counts) > 0       # drop empty pseudoautosomal copies
counts <- counts[keep_g, ]; gene_name <- gene_name[keep_g]

dx <- s$dx_clean; hit <- dx %in% names(RECODE); dx[hit] <- unname(RECODE[dx[hit]]); s$diagnosis <- dx

if (USE_ONE_PER_PATIENT) { k <- s$one_per_patient == "True"; counts <- counts[, k]; s <- s[k, ] }

n_dx  <- table(s$diagnosis)
small <- names(n_dx)[n_dx < MIN_PER_GROUP]
if (length(small)) {
  cat("left out (fewer than", MIN_PER_GROUP, "samples):", paste0(small, " (n=", n_dx[small], ")"), "\n")
  k <- !(s$diagnosis %in% small); counts <- counts[, k]; s <- s[k, ]
}
groups <- c(GROUP_ORDER[GROUP_ORDER %in% s$diagnosis], sort(setdiff(unique(s$diagnosis), GROUP_ORDER)))
gname  <- make.names(groups)                 # syntax-safe names used inside the model
stopifnot(length(groups) >= 2)
s$dx_f    <- factor(make.names(s$diagnosis), levels = gname)
s$batch_f <- factor(make.names(s$batch))
patients_with_2_dx <- sum(tapply(s$diagnosis, s$patient_key, function(x) length(unique(x))) > 1)
print(table(diagnosis = s$diagnosis))
c(samples = ncol(counts), patients = length(unique(s$patient_key)), genes = nrow(counts),
  patients_with_more_than_one_sample = sum(table(s$patient_key) > 1), patients_with_two_diagnoses = patients_with_2_dx)

## Batch against diagnosis
Rows are batches, columns are diagnoses. Read this before the results: a comparison is within-batch only where a batch holds samples of both groups.

In [ ]:
table(batch = s$batch, diagnosis = s$diagnosis)

## Design and gene filter
`~ 0 + diagnosis + batch`. If a batch cannot be separated from the diagnoses (for example, a batch that holds a single diagnosis found nowhere else), its column is dropped and printed, so the model can be fitted. `filterByExpr` keeps genes with enough counts to be informative, and `calcNormFactors` (TMM) corrects for library composition.

In [ ]:
design <- model.matrix(~ 0 + dx_f + batch_f, s)
colnames(design) <- sub("^dx_f", "", colnames(design))
colnames(design) <- sub("^batch_f", "batch_", colnames(design))
q <- qr(design)
if (q$rank < ncol(design)) {                   # diagnosis columns come first, so only batch columns are dropped
  keep_cols <- sort(q$pivot[seq_len(q$rank)])
  cat("batch columns dropped (not separable from the diagnoses):", colnames(design)[-keep_cols], "\n")
  design <- design[, keep_cols, drop = FALSE]
}
stopifnot(all(gname %in% colnames(design)))

dge  <- DGEList(counts, genes = data.frame(gene_id = rownames(counts), gene_name = gene_name,
                                           row.names = rownames(counts), stringsAsFactors = FALSE))
keep <- filterByExpr(dge, design)
dge  <- calcNormFactors(dge[keep, , keep.lib.sizes = FALSE])
c(coefficients = ncol(design), genes_kept = sum(keep), genes_total = length(keep))

## voom with the patient as a block
`voom` turns counts into log2 counts per million with precision weights. The patient correlation is estimated, used in a second voom, and estimated again; the second estimate goes into the model fit. If no patient has more than one sample (for example with `USE_ONE_PER_PATIENT = TRUE`), no block is used.

In [ ]:
use_block <- any(table(s$patient_key) > 1)
if (use_block) {
  v    <- voom(dge, design)
  cor1 <- duplicateCorrelation(v, design, block = s$patient_key)$consensus
  v    <- voom(dge, design, block = s$patient_key, correlation = cor1)
  cor2 <- duplicateCorrelation(v, design, block = s$patient_key)$consensus
  fit  <- lmFit(v, design, block = s$patient_key, correlation = cor2)
  round(c(first_estimate = cor1, second_estimate = cor2), 3)
} else {
  v   <- voom(dge, design)
  fit <- lmFit(v, design)
  cat("no patient has more than one sample: fitted without a patient block\n")
}

## The comparisons
Every pair of diagnoses; each contrast is the later group minus the earlier one in `GROUP_ORDER`, so against `HC` a positive log fold change means higher in the disease. `batches_with_both_groups` is the number of batches holding samples of both groups: 0 means that comparison is completely tangled with batch.

In [ ]:
pairs <- combn(gname, 2, simplify = FALSE)
lab   <- setNames(groups, gname)
safe  <- function(x) gsub("[^A-Za-z0-9]+", "-", x)
nm    <- sapply(pairs, function(p) sprintf("%s_vs_%s", safe(lab[[p[2]]]), safe(lab[[p[1]]])))
cm    <- makeContrasts(contrasts = sapply(pairs, function(p) paste0(p[2], "-", p[1])), levels = design)
colnames(cm) <- nm
fit2  <- eBayes(contrasts.fit(fit, cm))
tabs  <- lapply(setNames(nm, nm), function(k) topTable(fit2, coef = k, number = Inf, sort.by = "P"))

both <- function(a, b) sum(tapply(s$diagnosis == a, s$batch, any) & tapply(s$diagnosis == b, s$batch, any))
summary_tab <- data.frame(
  comparison = nm,
  n_group1 = sapply(pairs, function(p) sum(s$diagnosis == lab[[p[2]]])),
  n_group2 = sapply(pairs, function(p) sum(s$diagnosis == lab[[p[1]]])),
  batches_with_both_groups = sapply(pairs, function(p) both(lab[[p[2]]], lab[[p[1]]])),
  adj_p_below_cutoff = sapply(tabs, function(t) sum(t$adj.P.Val < ALPHA)),
  top_genes_up   = sapply(tabs, function(t) sum(t$adj.P.Val < ALPHA & t$logFC >=  LFC)),
  top_genes_down = sapply(tabs, function(t) sum(t$adj.P.Val < ALPHA & t$logFC <= -LFC)),
  row.names = NULL)
summary_tab$note <- ifelse(summary_tab$batches_with_both_groups == 0,
                           "CONFOUNDED WITH BATCH: no batch holds both groups", "")
summary_tab

## Save the tables and look at the top genes

In [ ]:
top <- lapply(tabs, function(t) t[t$adj.P.Val < ALPHA & abs(t$logFC) >= LFC, c("gene_id", "gene_name", "logFC", "AveExpr", "adj.P.Val")])
for (k in names(tabs)) {
  write.csv(tabs[[k]], file.path(OUT_DIR, sprintf("de_%s.csv", k)), row.names = FALSE)
  write.csv(top[[k]],  file.path(OUT_DIR, sprintf("top_genes_%s.csv", k)), row.names = FALSE)
}
write.csv(summary_tab, file.path(OUT_DIR, "summary_comparisons.csv"), row.names = FALSE)
for (k in names(top)) {
  cat("\n", k, ": ", nrow(top[[k]]), " top genes; the 5 with the smallest adjusted p\n", sep = "")
  if (nrow(top[[k]])) print(head(transform(top[[k]], logFC = round(logFC, 2), AveExpr = round(AveExpr, 1),
                                           adj.P.Val = signif(adj.P.Val, 2)), 5), row.names = FALSE)
}
if (any(summary_tab$batches_with_both_groups == 0))
  cat("\nWARNING: comparisons marked CONFOUNDED WITH BATCH cannot be told apart from facility; do not read them as disease effects.\n")
c(genes_in_any_top_list = length(unique(unlist(lapply(top, `[[`, "gene_id")))))
list.files(OUT_DIR)

---
## What to check
1. **Batch against diagnosis** (above): which diagnoses share a batch with the healthy controls?
2. **`batches_with_both_groups`** in the summary: trust a healthy-control comparison more when this is higher; treat 0 as unusable.
3. **The interferon genes.** In CANDLE, SAVI, AGS and UAID against `HC`, the type I interferon response genes (for example *IFI27, IFI44L, ISG15, RSAD2, MX1*) are expected to be up. Look for them in `top_genes_<disease>_vs_HC.csv`; if they are absent or go the other way, something about the cohort or batch is wrong.
4. If the healthy controls are only in a few batches, say so wherever you report these results.

## References
1. Ritchie ME, Phipson B, Wu D, Hu Y, Law CW, Shi W, Smyth GK. limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Res* 2015;43:e47.
2. Law CW, Chen Y, Shi W, Smyth GK. voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biol* 2014;15:R29.
3. Smyth GK, Michaud J, Scott HS. Use of within-array replicate spots for assessing differential expression in microarray experiments. *Bioinformatics* 2005;21:2067-2075.
4. Robinson MD, McCarthy DJ, Smyth GK. edgeR. *Bioinformatics* 2010;26:139-140.
5. Robinson MD, Oshlack A. A scaling normalization method for differential expression analysis of RNA-seq data. *Genome Biol* 2010;11:R25.
6. Chen Y, Lun ATL, Smyth GK. From reads to genes to pathways. *F1000Research* 2016;5:1438.
7. Benjamini Y, Hochberg Y. Controlling the false discovery rate. *J R Stat Soc B* 1995;57:289-300.
8. Love MI, Anders S, Kim V, Huber W. RNA-seq workflow: gene-level exploratory analysis and differential expression. *F1000Research* 2015;4:1070.